# Week 2 Day 5: can the improver make the claim agent cheaper?

My version of Ed's `labs/week2day5.ipynb`, cut down to what my two experiments need. The README beside this notebook explains them and the results. The judge, the claim agent, the dispatcher, the scoring, the rule for keeping a version, the improver and its recipe are Ed's, with these changes:

- **Two more choices in the configuration.** `wording` is `full`, `short` (Ed's `data/policy_wording_short.md`) or `short_v2` (my rewrite from Week 2 Day 1). `sections` is `all`, or `needed` for only the sections a claim needs. `make_agent` builds the prompt for each claim.
- **Two more measures.** Each run also records the prompt tokens a claim and the share of them read from the prompt cache. Ed's rule for keeping a version doesn't use them.
- **Runs of each choice without the improver.** Before the improver starts, version 1 runs with each choice, so the improver's picks can be checked against them. The improver never reads these runs, and they're paid apart from its budget.
- **The improver's tools and recipe** include the wording and the sections. `show_proposal` prints "none given" when the improver leaves out a reason, where Ed's stops with a `KeyError`.
- **Three rounds** in place of Ed's two, each in one cell. `run_round` shows each proposal and asks you to approve or reject it, in place of Ed's cells to run in turn.
- **Experiment 2** is Ed's stretch challenge from the free version of this lab.

Left out from Ed's lab: his explanations, the check of the judge on CLM-1001, the table of version 1's wrong claims, and the setup for the product.

Both experiments take about 25 minutes and cost about 50 cents. The improver makes its own choices, so read each proposal before you approve it.

In [ ]:
import difflib
import json
import operator
import os
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, Send
from openrouter import OpenRouter

# This notebook sits three folders below the repo, so the payout rules come from labs/
ROOT = Path("../../..")
sys.path.insert(0, str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")


langfuse = get_client()
langfuse_handler = CallbackHandler()

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## The claims and the wordings

`TINY = True` keeps 5 claims of each set, for a quick check.

In [ ]:
DATA = ROOT / "data"


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


TINY = False

evaluation_claims, evaluation_labels = get_sample_data(tiny=TINY)
held_out_claims, held_out_labels = get_sample_data(tiny=TINY, held_out=True)

claims = {claim["claim_id"]: claim for claim in evaluation_claims + held_out_claims}
labels = evaluation_labels | held_out_labels
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

SETS = {"evaluation": list(evaluation_labels), "held-out": list(held_out_labels)}
print(f"{len(SETS['evaluation'])} evaluation claims and {len(SETS['held-out'])} held-out claims")

In [ ]:
V2_PATH = Path("../week2day1_models_and_wording/week2day1_policy_wording_short_v2.md")


def without_comments(text):
    """The text with its HTML comments taken out."""
    return re.sub(r"<!--.*?-->\s*", "", text, flags=re.DOTALL)


WORDINGS = {
    "full": (DATA / "policy_wording.md").read_text(encoding="utf-8"),
    "short": (DATA / "policy_wording_short.md").read_text(encoding="utf-8"),
    "short_v2": without_comments(V2_PATH.read_text(encoding="utf-8")),
}
NUMBERED = ["full", "short_v2"]

for name, text in WORDINGS.items():
    print(f"{name}: {len(text.split()):,} words")

In [ ]:
BENEFIT_SECTION = {
    "trip_cancellation": "2",
    "trip_interruption": "2",
    "trip_delay": "3",
    "baggage_delay": "4",
    "baggage": "5",
    "passport": "6",
    "emergency_medical": "7",
}
EVERY_CLAIM_SECTIONS = ["1", "8", "9", "10", "11", "12"]


def split_numbered_sections(wording):
    """The opening of the wording, and its sections keyed by their number."""
    opening, *parts = wording.split("\n## ")
    return opening, {part.split(".")[0]: "## " + part for part in parts}


def wording_for(config, claim):
    """The policy wording a claim's prompt gets under a configuration."""
    wording = WORDINGS[config["wording"]]
    if config["sections"] == "all":
        return wording
    opening, sections = split_numbered_sections(wording)
    wanted = sorted({BENEFIT_SECTION[claim["claim_type"]], *EVERY_CLAIM_SECTIONS}, key=int)
    return "\n".join([opening.rstrip() + "\n", *(sections[number].rstrip() + "\n" for number in wanted)])


for name in NUMBERED:
    assert sorted(split_numbered_sections(WORDINGS[name])[1], key=int) == [str(n) for n in range(1, 13)], name

NEEDED_WORDS = {}
for name in NUMBERED:
    counts = [len(wording_for({"wording": name, "sections": "needed"}, claim).split()) for claim in claims.values()]
    NEEDED_WORDS[name] = (min(counts), max(counts))
    print(f"{name}, needed sections: {min(counts):,} to {max(counts):,} words a claim, against {len(WORDINGS[name].split()):,}")

## The judge, the claim agent and the scoring

Ed's, with the wording and sections in the prompt and the tokens counted in `work_claim`.

In [ ]:
JEV = "typesafe/jev-1.13"

# Jev numbers the grades by their position in the list, so F is 0 and A is 3
RUBRIC = {
    "F": "different grounds from the handler's reason, or it contradicts the handler's reason",
    "C": "partly the same as the handler's reason: one of the handler's grounds or the section is missing, or it adds a ground the handler did not give",
    "B": "the same grounds and the same section of the policy wording as the handler's reason",
    "A": "the same grounds and section as the handler's reason, and clearly better expressed: clearer for the customer, with the section and the figures, shorter, and no jargon",
}
GRADES = list(RUBRIC)

JUDGE_QUESTIONS = {
    "grade": {
        "type": "score",
        "instructions": "An expert claims handler decided a travel insurance claim and gave `handler.reason`, resting on `handler.reference`. Grade how well `agent.rationale` explains the same decision.",
        "criteria": [f"{letter}: {text}" for letter, text in RUBRIC.items()],
    },
    "same_grounds": {"type": "noul", "instructions": "`agent.rationale` gives the same main grounds for the decision as `handler.reason`"},
    "same_section": {"type": "noul", "instructions": "`agent.rationale` refers to the same section of the policy wording as `handler.reference`"},
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def judge(rationale, decision, expected):
    """Ask Jev to grade a rationale against the handler's reason."""
    state = {
        "handler": {"decision": expected["decision"], "reason": expected["reason"], "reference": expected["reference"]},
        "agent": {"decision": decision, "rationale": rationale},
    }
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions=JUDGE_QUESTIONS)
    answers = response.answers
    probabilities = answers["grade"].probabilities
    most_likely = max(probabilities, key=probabilities.get)
    return {
        "grade": answers["grade"].score,
        "letter": GRADES[int(most_likely)],
        "same_grounds": answers["same_grounds"].noul,
        "same_section": answers["same_section"].noul,
        "judge_cost": response.usage.cost,
    }

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
Use the payout tool to work out the payout on an approved claim, and record the payout it returns.
"""

versions = [{"version": 1, "instructions": INSTRUCTIONS, "model": "openai/gpt-6-luna", "reasoning_effort": "low",
             "wording": "full", "sections": "all"}]
current = 1

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_agent(config, claim):
    """Build the claim agent for one version of the configuration and one claim."""
    model = ChatOpenAI(model=config["model"], base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                       reasoning_effort=config["reasoning_effort"], timeout=120, max_retries=5)
    return create_agent(model=model, tools=[record_decision, calculate_payout],
                        system_prompt=f"{config['instructions']}\n{wording_for(config, claim)}\n")

In [ ]:
class Dispatch(TypedDict):
    """What the dispatcher keeps for one run."""

    config: dict
    claims: list[str]
    results: Annotated[list, operator.add]


def fan_out(state: Dispatch):
    """Send each claim to a claim agent, with the configuration to use."""
    return [Send("work_claim", {"claim_id": claim_id, "config": state["config"]}) for claim_id in state["claims"]]


def work_claim(task):
    """Run a claim agent on one claim and return its result."""
    claim_id = task["claim_id"]
    agent = make_agent(task["config"], claims[claim_id])
    started = datetime.now()
    result = agent.invoke({"messages": [{"role": "user", "content": make_user_prompt(claims[claim_id])}]})
    answer = recorded.get(claim_id, {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    usage = [message.response_metadata["token_usage"] for message in result["messages"] if message.type == "ai"]
    cost = sum(call["cost"] for call in usage)
    prompt_tokens = sum(call["prompt_tokens"] for call in usage)
    cached_tokens = sum((call.get("prompt_tokens_details") or {}).get("cached_tokens") or 0 for call in usage)
    return {"results": [{"claim_id": claim_id, **answer, "cost": cost, "prompt_tokens": prompt_tokens,
                         "cached_tokens": cached_tokens, "started": started, "finished": datetime.now()}]}


builder = StateGraph(Dispatch)
builder.add_node("work_claim", work_claim, retry_policy=RetryPolicy(max_attempts=3))
builder.add_conditional_edges(START, fan_out, ["work_claim"])
builder.add_edge("work_claim", END)
dispatcher = builder.compile()

In [ ]:
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def score_table(results):
    """One row for each claim: the agent's answer beside the handler's, with the judge's grade, cost, tokens and time."""
    rows = []
    for answer in results:
        expected = labels[answer["claim_id"]]
        rows.append({
            "claim_id": answer["claim_id"],
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            **judge(answer["rationale"], answer["decision"], expected),
            "llm_cost": answer["cost"],
            "prompt_tokens": answer["prompt_tokens"],
            "cached_tokens": answer["cached_tokens"],
            "seconds": (answer["finished"] - answer["started"]).total_seconds(),
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run: quality, cost, tokens and time."""
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "grade": table["grade"].mean(),
        "cost_per_claim": table["llm_cost"].mean(),
        "seconds_per_claim": table["seconds"].mean(),
        "prompt_tokens_per_claim": table["prompt_tokens"].mean(),
        "cached_share": table["cached_tokens"].sum() / table["prompt_tokens"].sum(),
    }


SCORE_NAMES = ["decision_accuracy", "accuracy", "cost_of_mistakes", "grade", "cost_per_claim", "seconds_per_claim",
               "prompt_tokens_per_claim", "cached_share"]

In [ ]:
tables = {}
scoreboard = []
spent = 0.0
control_spent = 0.0
EXPERIMENT = "wording"


def slot_name(slot):
    """A model slot as text, such as openai/gpt-6-luna at low."""
    return f"{slot['model']} at {slot['reasoning_effort']}" if slot else ""


def evaluate(config, set_name, control=False):
    """Run one version of the configuration on one set, score it and add it to the scoreboard."""
    global spent, control_spent
    recorded.clear()
    run_name = f"Version {config['version']} on the {set_name} set"
    result = dispatcher.invoke({"config": config, "claims": SETS[set_name]},
                               {"max_concurrency": 25, "callbacks": [langfuse_handler], "run_name": run_name})
    table = score_table(result["results"])
    scores = run_scores(table)
    cost = table["llm_cost"].sum() + table["judge_cost"].sum()
    if control:
        control_spent += cost
    else:
        spent += cost
    tables[(config["version"], set_name)] = table
    scoreboard.append({"experiment": EXPERIMENT, "version": config["version"], "set": set_name, "model": config["model"],
                       "reasoning_effort": config["reasoning_effort"], "hard": slot_name(config.get("hard")),
                       "wording": config["wording"], "sections": config["sections"], "control": control, **scores})
    print(f"{run_name}: accuracy {scores['accuracy']:.2f}, grade {scores['grade']:.2f}, "
          f"${scores['cost_per_claim']:.5f}, {scores['seconds_per_claim']:.1f} seconds and "
          f"{scores['prompt_tokens_per_claim']:,.0f} prompt tokens a claim, {scores['cached_share']:.0%} from the cache")
    return scores

## Experiment 1: a shorter prompt

### Version 1

Once on the evaluation set and twice on the held-out set, so we know how much the scores move with no change.

In [ ]:
evaluate(versions[0], "evaluation")
evaluate(versions[0], "held-out")
evaluate(versions[0], "held-out")

pd.DataFrame(scoreboard).round(4)

### The rule for keeping a version

Ed's objectives, noise, tolerance, budget and models allowed.

In [ ]:
OBJECTIVES = {"accuracy": 0.6, "grade": 1.0}
NOISE = {"accuracy": 0.1, "grade": 0.15, "cost_per_claim": 0.15, "seconds_per_claim": 0.15}
TOLERANCE = {"accuracy": 0.1, "grade": 0.15, "cost_per_claim": 1.0, "seconds_per_claim": 0.25}
BUDGET = 0.50

ALLOWED_MODELS = {
    "openai/gpt-6-luna": {"reasoning_efforts": ["none", "low", "medium"], "price per million tokens": "$0.10 in, $0.50 out"},
    "google/gemini-3.5-flash-lite": {"reasoning_efforts": ["minimal", "low"], "price per million tokens": "$0.30 in, $2.50 out"},
    "deepseek/deepseek-v4.1-flash": {"reasoning_efforts": ["low"], "price per million tokens": "$0.30 in, $1.20 out"},
}


def compare(new, old):
    """Which scores got better by more than the noise, and which got worse by more than the tolerance."""
    better, worse = [], []
    for name in ["accuracy", "grade"]:
        if new[name] > old[name] + NOISE[name]:
            better.append(name)
        if new[name] < old[name] - TOLERANCE[name]:
            worse.append(name)
    for name in ["cost_per_claim", "seconds_per_claim"]:
        if new[name] < old[name] * (1 - NOISE[name]):
            better.append(name)
        if new[name] > old[name] * (1 + TOLERANCE[name]):
            worse.append(name)
    return better, worse


def keep_or_roll_back(new, old):
    """Decide whether to keep a new version from its held-out scores and the current version's average."""
    missed = [name for name, floor in OBJECTIVES.items() if new[name] < floor]
    better, worse = compare(new, old)
    if missed:
        return False, f"it missed the objectives on {', '.join(missed)}"
    if worse:
        return False, f"{', '.join(worse)} got worse"
    if not better:
        return False, "no score improved by more than the noise"
    return True, f"{', '.join(better)} improved"

### Each choice without the improver

Version 1 with one choice changed, and once with my short wording and only the sections needed together. Each runs once on the evaluation set and twice on the held-out set, like a new version.

In [ ]:
CONTROLS = [("short", "all"), ("short_v2", "all"), ("full", "needed"), ("short_v2", "needed")]

for wording, sections in CONTROLS:
    config = {**versions[0], "version": f"1, {wording}, {sections}", "wording": wording, "sections": sections}
    evaluate(config, "evaluation", control=True)
    evaluate(config, "held-out", control=True)
    evaluate(config, "held-out", control=True)

In [ ]:
def average(version, set_name):
    """The average of a version's scores over all its runs on a set."""
    runs = pd.DataFrame([row for row in scoreboard if row["version"] == version and row["set"] == set_name])
    return runs[SCORE_NAMES].mean().to_dict()


MEASURES = ["accuracy", "grade", "cost_per_claim", "seconds_per_claim", "prompt_tokens_per_claim", "cached_share"]
version_1 = average(1, "held-out")
rows = [{"version": "1", "wording": "full", "sections": "all", **{m: version_1[m] for m in MEASURES}, "rule would say": ""}]
for wording, sections in CONTROLS:
    scores = average(f"1, {wording}, {sections}", "held-out")
    keep, why = keep_or_roll_back(scores, version_1)
    rows.append({"version": "1", "wording": wording, "sections": sections, **{m: scores[m] for m in MEASURES},
                 "rule would say": f"{'keep' if keep else 'roll back'}: {why}"})

print("The average of two runs on the held-out set")
pd.DataFrame(rows).round(4)

### The improver

Ed's tools, with the wording and sections added, and his recipe with step 3 about them.

In [ ]:
rounds = []

LEVERS = {
    "wording": {
        "full": f"The full policy wording, {len(WORDINGS['full'].split()):,} words, sections numbered 1 to 12",
        "short": f"A concise rewrite of the full wording, {len(WORDINGS['short'].split()):,} words, sections not numbered",
        "short_v2": f"A concise rewrite that copies the definitions, thresholds, lists and examples word for word, "
                    f"{len(WORDINGS['short_v2'].split()):,} words, sections numbered 1 to 12",
    },
    "sections": {
        "all": "The whole wording goes into every claim's prompt",
        "needed": "Only the sections a claim needs go into its prompt, word for word: sections 1, 8, 9, 10, 11 and 12, "
                  "and the section for the claim's benefit. Only with a wording whose sections are numbered. "
                  + ", ".join(f"With {name}, {low:,} to {high:,} words a claim" for name, (low, high) in NEEDED_WORDS.items()),
    },
}


def latest(version, set_name):
    """The most recent scores for a version on a set."""
    return [row for row in scoreboard if row["version"] == version and row["set"] == set_name][-1]


def beside_handler(table):
    """Each claim in a table with the agent's answer beside the handler's and the judge's grade."""
    rows = []
    for row in table.to_dict("records"):
        expected = labels[row["claim_id"]]
        rows.append({
            "claim_id": row["claim_id"],
            "summary": claims[row["claim_id"]]["summary"],
            "agent": {"decision": row["agent"], "payout": row["agent_payout"], "rationale": row["rationale"]},
            "handler": {"decision": expected["decision"], "payout": expected["payout"], "reason": expected["reason"], "reference": expected["reference"]},
            "judge": {"grade": round(row["grade"], 2), "letter": row["letter"]},
        })
    return rows


def mistakes(version):
    """The evaluation claims a version got wrong, and the five right ones with the lowest grades."""
    table = tables[(version, "evaluation")]
    weakest = table[table["right"]].sort_values("grade").head(5)
    return {"wrong": beside_handler(table[~table["right"]]), "explained_least_well": beside_handler(weakest)}

In [ ]:
@tool
def read_results() -> str:
    """Read the current version of the claim agent's configuration, its scores, its mistakes and weakest explanations on the evaluation set, the earlier rounds, the objectives, the budget, the models allowed and the wordings and sections to choose from."""
    config = versions[current - 1]
    report = {
        "current_version": config,
        "scores": {set_name: average(current, set_name) for set_name in SETS},
        "evaluation_set": mistakes(current),
        "earlier_rounds": rounds,
        "objectives_on_the_held_out_set": OBJECTIVES,
        "noise": NOISE,
        "tolerance": TOLERANCE,
        "rule": "A new version is kept when it meets the objectives, at least one score improves by more than the noise and none gets worse by more than the tolerance",
        "budget_left": round(BUDGET - spent, 4),
        "models_allowed": ALLOWED_MODELS,
        "policy_wording_choices": LEVERS,
    }
    return json.dumps(report, indent=2, default=str)


@tool(parse_docstring=True)
def propose_change(instructions: str, model: str, reasoning_effort: str, wording: Literal["full", "short", "short_v2"],
                   sections: Literal["all", "needed"], reason: str) -> str:
    """Propose a new version of the claim agent's configuration. A human reviews it before it's saved.

    Args:
        instructions: The claim agent's complete instructions
        model: One of the models allowed
        reasoning_effort: A reasoning level allowed for that model
        wording: Which policy wording the claim agent gets
        sections: All the wording's sections, or only those the claim needs
        reason: Why this change should improve the scores
    """
    if reasoning_effort not in ALLOWED_MODELS.get(model, {}).get("reasoning_efforts", []):
        return f"{model} at {reasoning_effort} isn't allowed. Choose from the models allowed."
    if sections == "needed" and wording not in NUMBERED:
        return f"The {wording} wording doesn't number its sections, so it can only go in whole. Choose sections all, or another wording."
    version = len(versions) + 1
    versions.append({"version": version, "instructions": instructions, "model": model, "reasoning_effort": reasoning_effort,
                     "wording": wording, "sections": sections, "reason": reason})
    return f"Saved as version {version}. Run the evaluation next."


@tool
def run_evaluation() -> str:
    """Run the newest version once on the evaluation set and twice on the held-out set, then keep it or roll it back."""
    global current
    config = versions[-1]
    if config["version"] == current or any(r["version"] == config["version"] for r in rounds):
        return "There's no new version to evaluate. Propose a change first."
    estimate = 3 * len(SETS["held-out"]) * max(row["cost_per_claim"] for row in scoreboard)
    if spent + estimate > BUDGET:
        return f"Refused: the evaluation could cost about ${estimate:.2f} and ${BUDGET - spent:.2f} of the budget is left"
    before = average(current, "held-out")
    evaluate(config, "evaluation")
    evaluate(config, "held-out")
    evaluate(config, "held-out")
    after = average(config["version"], "held-out")
    keep, why = keep_or_roll_back(after, before)
    rounds.append({"round": len(rounds) + 1, "recipe_version": len(recipes), "version": config["version"],
                   "model": config["model"], "reasoning_effort": config["reasoning_effort"], "hard": slot_name(config.get("hard")),
                   "wording": config["wording"], "sections": config["sections"], "reason": config["reason"],
                   "kept": keep, "why": why, "held_out_before": before, "held_out_after": after})
    if keep:
        current = config["version"]
    verdict = "Kept" if keep else "Rolled back"
    scores = {"held_out_before": before, "held_out_after": after, "evaluation_after": latest(config["version"], "evaluation")}
    return f"{verdict} version {config['version']}: {why}. Version {current} is now the current version.\n{json.dumps(scores, indent=2, default=str)}"


recipes = []


@tool(parse_docstring=True)
def propose_recipe(recipe: str, reason: str) -> str:
    """Propose a new version of your recipe for improving the claim agent. A human reviews it before it's saved.

    Args:
        recipe: The complete new recipe
        reason: What this round showed, and why the new recipe should lead to better changes
    """
    recipes.append(recipe)
    return f"Saved as recipe version {len(recipes)}."

In [ ]:
RECIPE = """1. Call read_results to see how the current version is doing and which evaluation claims it got wrong or explained poorly.
2. For each of those claims, add a rule to the instructions that would make the agent get that claim right.
3. Call propose_change with the new instructions, keeping the same model and reasoning level. Choose the policy wording and sections you expect to score best.
4. Call run_evaluation to test the new version on both sets.
5. Improve this recipe, so that the next round makes a better change. Look at what happened in this round and in earlier rounds, then call propose_recipe with the new recipe. Keep this as the last step of the new recipe.
"""

recipes.append(RECIPE)

In [ ]:
improver_model = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                            reasoning_effort="medium", timeout=300, max_retries=5)

approval = HumanInTheLoopMiddleware(interrupt_on={
    "propose_change": {"allowed_decisions": ["approve", "reject"]},
    "propose_recipe": {"allowed_decisions": ["approve", "reject"]},
})


def make_improver(recipe):
    """Build the improver with a recipe as its instructions."""
    system_prompt = f"""You improve the configuration of UnderCover's claim agent, an AI agent that decides travel insurance claims.
The claim agent sees its instructions, the policy wording in the version and sections its configuration names, the customer's policy record and the claim. It doesn't see the claims manual or the customer's history.
Follow your recipe one step at a time.

# Your recipe

{recipe}
"""
    return create_agent(model=improver_model, tools=[read_results, propose_change, run_evaluation, propose_recipe],
                        system_prompt=system_prompt, middleware=[approval], checkpointer=InMemorySaver())

In [ ]:
def show_changed_lines(old, new):
    """Print the lines that differ between two texts."""
    for line in difflib.unified_diff(old.splitlines(), new.splitlines(), lineterm="", n=0):
        if not line.startswith(("---", "+++", "@@")):
            print(line)


def show_proposal(result):
    """Show the change the improver is waiting for us to approve."""
    if "__interrupt__" not in result:
        print("The improver finished without a proposal. Its last message:")
        print(result["messages"][-1].content)
        return
    request = result["__interrupt__"][0].value["action_requests"][0]
    args = request["args"]
    if request["name"] == "propose_change":
        old = versions[current - 1]
        print(f"Model: {old['model']} at {old['reasoning_effort']}, proposed {args['model']} at {args['reasoning_effort']}")
        print(f"Wording: {old['wording']} with {old['sections']} sections, proposed {args['wording']} with {args['sections']} sections\n")
        show_changed_lines(old["instructions"], args["instructions"])
    elif request["name"] == "propose_routing":
        old = versions[current - 1]
        print(f"Routine claims: {old['model']} at {old['reasoning_effort']}, proposed {args['routine_model']} at {args['routine_reasoning_effort']}")
        print(f"Hard claims: {slot_name(old.get('hard')) or 'the same as routine claims'}, proposed {args['hard_model']} at {args['hard_reasoning_effort']}")
    else:
        print(f"Recipe version {len(recipes)}, proposed version {len(recipes) + 1}\n")
        show_changed_lines(recipes[-1], args["recipe"])
    print(f"\nReason: {args.get('reason', 'none given')}")


APPROVE = Command(resume={"decisions": [{"type": "approve"}]})


def REJECT(reason):
    """Turn down the improver's proposal, with the reason it will read."""
    return Command(resume={"decisions": [{"type": "reject", "message": reason}]})


def show_rounds():
    """One row for each round: the version it tried, its wording, the held-out scores before and after, and the verdict."""
    rows = []
    for r in rounds:
        rows.append({
            "round": r["round"], "recipe": r["recipe_version"], "version": r["version"],
            "model": r["model"], "reasoning": r["reasoning_effort"], "hard": r.get("hard", ""), "wording": r["wording"], "sections": r["sections"],
            "accuracy_before": r["held_out_before"]["accuracy"], "accuracy_after": r["held_out_after"]["accuracy"],
            "grade_before": r["held_out_before"]["grade"], "grade_after": r["held_out_after"]["grade"],
            "tokens_before": r["held_out_before"]["prompt_tokens_per_claim"], "tokens_after": r["held_out_after"]["prompt_tokens_per_claim"],
            "kept": r["kept"], "why": r["why"],
        })
    return pd.DataFrame(rows).round(2)


APPROVE_ALL = False  # True approves every proposal without asking, for an unattended run
session = f"Improvement {datetime.now():%Y-%m-%d %H:%M}"


def round_config(name):
    """The run settings for one round: a thread of its own, in this notebook's Langfuse session."""
    return {"configurable": {"thread_id": name}, "callbacks": [langfuse_handler], "run_name": f"Improver {name}",
            "metadata": {"langfuse_session_id": session}}


def run_round(improver, config):
    """Run one round of the improver's recipe, showing each proposal and asking whether to approve it."""
    result = improver.invoke({"messages": [{"role": "user", "content": "Please carry out one round of your recipe."}]}, config)
    pauses = 0
    while "__interrupt__" in result and pauses < 10:
        pauses += 1
        show_proposal(result)
        answer = "" if APPROVE_ALL else input("Press Enter to approve, or type a reason to reject: ").strip()
        print("Approved" if not answer else f"Rejected: {answer}")
        result = improver.invoke(REJECT(answer) if answer else APPROVE, config)
    print(result["messages"][-1].content)

### Rounds

Each round runs the improver's recipe once. It stops at every proposal, first a change and then a new recipe, and asks you about it: press Enter to approve, or type a reason to reject it, which the improver reads. Set `APPROVE_ALL = True` above to approve everything without asking, as in my runs.

In [ ]:
# Round 1: the improver reads the results, proposes a change, then a new recipe
run_round(make_improver(recipes[-1]), round_config("round-1"))

In [ ]:
# Round 2, with the newest recipe
run_round(make_improver(recipes[-1]), round_config("round-2"))

In [ ]:
# Round 3, with the newest recipe
run_round(make_improver(recipes[-1]), round_config("round-3"))

### The results of experiment 1

In [ ]:
pd.DataFrame(scoreboard).round(4)

In [ ]:
show_rounds()


print(f"Current version: {current}, with the {versions[current - 1]['wording']} wording and {versions[current - 1]['sections']} sections")
print(f"Spent by the improver on evaluations: ${spent:.4f} of ${BUDGET:.2f}")
print(f"Spent on the levers alone: ${control_spent:.4f}")

## Experiment 2: a cheaper model for routine claims

Ed's stretch challenge from the free version of this lab: a cheap model for routine claims, a stronger one for the rest, and a rule in code that routes each claim. The rule sends baggage and trip delay claims to the stronger model, since in one of my runs they were all of version 1's mistakes on the evaluation set. The instructions and the wording stay fixed, so the improver chooses only the models. The tolerance on time goes up to 200%, and the improver's budget starts again.

In [ ]:
HARD_TYPES = {"baggage", "trip_delay"}


def is_hard(claim):
    """True for the claim types where version 1 made all its mistakes on the evaluation set."""
    return claim["claim_type"] in HARD_TYPES


for set_name, claim_ids in SETS.items():
    hard = sum(is_hard(claims[claim_id]) for claim_id in claim_ids)
    print(f"The {set_name} set: {hard} hard claims and {len(claim_ids) - hard} routine ones")


def make_agent(config, claim):
    """Build the claim agent for one claim, on the hard slot's model for a hard claim and the routine slot's otherwise."""
    slot = config["hard"] if config.get("hard") and is_hard(claim) else config
    model = ChatOpenAI(model=slot["model"], base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                       reasoning_effort=slot["reasoning_effort"], timeout=120, max_retries=5)
    return create_agent(model=model, tools=[record_decision, calculate_payout],
                        system_prompt=f"{config['instructions']}\n{wording_for(config, claim)}\n")


def by_group(table):
    """A run's scores on the routine claims and on the hard ones."""
    groups = ["hard" if is_hard(claims[claim_id]) else "routine" for claim_id in table["claim_id"]]
    scores = table.assign(group=groups).groupby("group").agg(
        claims=("claim_id", "size"), accuracy=("right", "mean"), grade=("grade", "mean"),
        cost_per_claim=("llm_cost", "mean"), seconds_per_claim=("seconds", "mean"))
    return scores.round(5).to_dict("index")

In [ ]:
wording_experiment = {"versions": versions, "current": current, "rounds": rounds, "recipes": recipes, "scoreboard": scoreboard, "spent": spent}

EXPERIMENT = "routing"
TOLERANCE = {**TOLERANCE, "seconds_per_claim": 2.0}
versions = [{"version": 1, "instructions": INSTRUCTIONS, "model": "openai/gpt-6-luna", "reasoning_effort": "low", "hard": None,
             "wording": "short_v2", "sections": "needed"}]
current = 1
rounds, scoreboard, tables = [], [], {}
spent = 0.0

evaluate(versions[0], "evaluation")
evaluate(versions[0], "held-out")
evaluate(versions[0], "held-out")

### Each setup without the improver

In [ ]:
LUNA = "openai/gpt-6-luna"
ROUTING_CONTROLS = {
    "1, medium for every claim": {"reasoning_effort": "medium", "hard": None},
    "1, low routine, medium hard": {"hard": {"model": LUNA, "reasoning_effort": "medium"}},
    "1, none routine, medium hard": {"reasoning_effort": "none", "hard": {"model": LUNA, "reasoning_effort": "medium"}},
}

for name, change in ROUTING_CONTROLS.items():
    config = {**versions[0], "version": name, **change}
    evaluate(config, "evaluation", control=True)
    evaluate(config, "held-out", control=True)
    evaluate(config, "held-out", control=True)

In [ ]:
start = average(1, "held-out")
rows = [{"version": "1, low for every claim", **{m: start[m] for m in MEASURES}, "rule would say": ""}]
for name in ROUTING_CONTROLS:
    scores = average(name, "held-out")
    keep, why = keep_or_roll_back(scores, start)
    rows.append({"version": name, **{m: scores[m] for m in MEASURES}, "rule would say": f"{'keep' if keep else 'roll back'}: {why}"})

print("The average of two runs on the held-out set")
pd.DataFrame(rows).round(4)

### The improver for this experiment

It reads the scores on routine and hard claims, and proposes a model and reasoning level for each.

In [ ]:
ROUTING = (f"Claims of the types {', '.join(sorted(HARD_TYPES))} are hard and go to the hard slot's model. "
           "Every other claim is routine and goes to the routine slot's model. This rule is fixed.")


@tool
def read_routing_results() -> str:
    """Read the current version of the claim agent's configuration, its scores overall and on routine and hard claims, the earlier rounds, the objectives, the budget and the models allowed."""
    report = {
        "current_version": versions[current - 1],
        "routing": ROUTING,
        "scores": {set_name: average(current, set_name) for set_name in SETS},
        "evaluation_set_by_group": by_group(tables[(current, "evaluation")]),
        "earlier_rounds": rounds,
        "objectives_on_the_held_out_set": OBJECTIVES,
        "noise": NOISE,
        "tolerance": TOLERANCE,
        "rule": "A new version is kept when it meets the objectives, at least one score improves by more than the noise and none gets worse by more than the tolerance",
        "budget_left": round(BUDGET - spent, 4),
        "models_allowed": ALLOWED_MODELS,
    }
    return json.dumps(report, indent=2, default=str)


@tool(parse_docstring=True)
def propose_routing(routine_model: str, routine_reasoning_effort: str, hard_model: str, hard_reasoning_effort: str, reason: str) -> str:
    """Propose a new version of the claim agent's configuration, with a model for routine claims and one for hard claims. A human reviews it before it's saved.

    Args:
        routine_model: One of the models allowed, for routine claims
        routine_reasoning_effort: A reasoning level allowed for that model
        hard_model: One of the models allowed, for hard claims
        hard_reasoning_effort: A reasoning level allowed for that model
        reason: Why this change should improve the scores
    """
    for model, effort in [(routine_model, routine_reasoning_effort), (hard_model, hard_reasoning_effort)]:
        if effort not in ALLOWED_MODELS.get(model, {}).get("reasoning_efforts", []):
            return f"{model} at {effort} isn't allowed. Choose from the models allowed."
    version = len(versions) + 1
    versions.append({**versions[0], "version": version, "model": routine_model, "reasoning_effort": routine_reasoning_effort,
                     "hard": {"model": hard_model, "reasoning_effort": hard_reasoning_effort}, "reason": reason})
    return f"Saved as version {version}. Run the evaluation next."

In [ ]:
RECIPE_ROUTING = """1. Call read_routing_results to see how the current version scores overall and on routine and hard claims, with its cost and time.
2. Decide what to change. A cheaper model or a lower reasoning level can serve the claims the agent already gets right, and a stronger one the claims it gets wrong.
3. Call propose_routing with the model and reasoning level for routine claims and for hard claims.
4. Call run_evaluation to test the new version on both sets.
5. Improve this recipe, so that the next round makes a better change. Look at what happened in this round and in earlier rounds, then call propose_recipe with the new recipe. Keep this as the last step of the new recipe.
"""

recipes = [RECIPE_ROUTING]

routing_approval = HumanInTheLoopMiddleware(interrupt_on={
    "propose_routing": {"allowed_decisions": ["approve", "reject"]},
    "propose_recipe": {"allowed_decisions": ["approve", "reject"]},
})


def make_routing_improver(recipe):
    """Build the improver for this part, with a recipe as its instructions."""
    system_prompt = f"""You improve the configuration of UnderCover's claim agent, an AI agent that decides travel insurance claims.
Its instructions and its policy wording are fixed. You choose the model and reasoning level for routine claims and for hard claims.
Follow your recipe one step at a time.

# Your recipe

{recipe}
"""
    return create_agent(model=improver_model, tools=[read_routing_results, propose_routing, run_evaluation, propose_recipe],
                        system_prompt=system_prompt, middleware=[routing_approval], checkpointer=InMemorySaver())

### Rounds

As in experiment 1, with this experiment's improver.

In [ ]:
# Routing round 1: the improver proposes a model for each slot, then a new recipe
run_round(make_routing_improver(recipes[-1]), round_config("routing-round-1"))

In [ ]:
# Routing round 2, with the newest recipe
run_round(make_routing_improver(recipes[-1]), round_config("routing-round-2"))

### The results of experiment 2

The scoreboard of both experiments is saved beside this notebook as `week2day5_scoreboard.csv`.

In [ ]:
pd.DataFrame(scoreboard).round(4)

In [ ]:
show_rounds()


config = versions[current - 1]
print(f"Current version: {current}, with {config['model']} at {config['reasoning_effort']} for routine claims "
      f"and {slot_name(config['hard']) or 'the same for hard claims'}")
print(f"Spent by the improver on evaluations: ${spent:.4f} of ${BUDGET:.2f}")
print(f"Spent on the runs of each choice alone, in both parts: ${control_spent:.4f}")

In [ ]:
pd.DataFrame(wording_experiment["scoreboard"] + scoreboard).to_csv("week2day5_scoreboard.csv", index=False)